In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Papa"
STUDENT_ID = "TUPM-26-1927"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR.parent / "data"
OUTPUT_DIR = WORK_DIR

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

for path in [measurement_path, label_path]:
    assert path.is_file(), f"Missing file: {path}"

raw_measurements = np.loadtxt(measurement_path)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert len(labels) == len(raw_measurements)

print("Workspace:", WORK_DIR.resolve())
print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Measurements:", raw_measurements.shape)
print("Labels:", labels.shape)

Workspace: C:\Users\Cyrus\OneDrive\Desktop\Lab7_NumPy_Pandas\Lab7_NumPy_Pandas_Papa\Lab7_NumPy_Pandas_Additional_Exercises
Seed Digit: 7 | ID Sum: 27
Random Seed: 261927
Measurements: (1567, 590)
Labels: (1567,)


In [15]:
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

usable_indices = np.flatnonzero(~(all_missing | constant))

n_select = 5 + (SEED_DIGIT % 3)
rng = np.random.default_rng(RANDOM_SEED)
selected_indices = np.sort(
    rng.choice(usable_indices, size=n_select, replace=False)
)
selected_names = [f"Feature_{i:03d}" for i in selected_indices]

print("Features With Any Missing:", missing.any(axis=0).sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))
print("Features To Select:", n_select)
print("Selected Features:", selected_names)

Features With Any Missing: 538
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474
Features To Select: 6
Selected Features: ['Feature_000', 'Feature_044', 'Feature_051', 'Feature_064', 'Feature_174', 'Feature_556']


In [16]:
selected = raw_measurements[:, selected_indices]
sel_missing = np.isnan(selected)

means = np.nanmean(selected, axis=0)
medians = np.nanmedian(selected, axis=0)
sds = np.nanstd(selected, axis=0, ddof=1)
missing_pct = 100 * sel_missing.mean(axis=0)

mad = np.nanmedian(np.abs(selected - medians), axis=0)
safe_scale = np.where(mad > 0, 1.4826 * mad, 1.0)

robust_z = (selected - medians) / safe_scale
flags = np.abs(robust_z) > 3.5

print("Means:", means.round(4))
print("Medians:", medians.round(4))
print("Sample SDs:", sds.round(4))
print("Missing %:", missing_pct.round(2))
print("MAD:", mad.round(4))
print("Zero-MAD Features:", (mad == 0).sum())
print("Total Flagged:", flags.sum())

Means: [3.0144529e+03 1.0031200e+01 1.5742100e+02 2.0542100e+01 3.2010000e-01
 4.2169000e+00]
Medians: [3.011490e+03 1.003480e+01 1.833182e+02 2.002130e+01 3.238000e-01
 4.067100e+00]
Sample SDs: [7.36218e+01 1.75000e-01 6.09251e+01 4.97750e+00 7.12000e-02 1.28000e+00]
Missing %: [ 0.38  0.06  0.06  0.45  0.06 16.59]
MAD: [45.23    0.1143 41.7907  2.7118  0.0469  0.4427]
Zero-MAD Features: 0
Total Flagged: 59


In [17]:
flag_counts = flags.sum(axis=0)
observed_counts = (~sel_missing).sum(axis=0)
flag_pct = 100 * flag_counts / observed_counts

quality = pd.DataFrame({
    "Feature": selected_names,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "Missing_Pct": missing_pct,
    "MAD": mad,
    "Flagged_Count": flag_counts,
    "Flagged_Pct": flag_pct,
}).sort_values(
    ["Flagged_Pct", "Feature"], ascending=[False, True]
).reset_index(drop=True)

quality.insert(0, "Rank", quality.index + 1)

quality_path = OUTPUT_DIR / "measurement_quality.csv"
quality.to_csv(quality_path, index=False)

# Verify the vectorized results with Pandas and a plain loop
check = pd.DataFrame(selected)
assert np.allclose(check.mean().to_numpy(), means)
assert np.allclose(check.median().to_numpy(), medians)
assert np.allclose(check.std().to_numpy(), sds)

for j in range(selected.shape[1]):
    col = selected[:, j]
    col = col[~np.isnan(col)]
    med = np.median(col)
    m = np.median(np.abs(col - med))
    scale = 1.4826 * m if m > 0 else 1.0
    loop_count = sum(abs((v - med) / scale) > 3.5 for v in col)
    assert loop_count == flag_counts[j]

verified = True

print(quality.round(4))
print("Selected Features:", selected_names)
print("Highest Missing %:", f"{missing_pct.max():.2f}",
      "(", selected_names[missing_pct.argmax()], ")")
print("Highest Flagged % Feature:", quality.loc[0, "Feature"])
print("Total Flagged Observations:", flag_counts.sum())
print("Vectorized Results Verified:", verified)
print("File Created:", quality_path.name, quality_path.is_file())

   Rank      Feature       Mean     Median  Sample_SD  Missing_Pct      MAD  \
0     1  Feature_064    20.5421    20.0213     4.9775       0.4467   2.7118   
1     2  Feature_556     4.2169     4.0671     1.2800      16.5922   0.4427   
2     3  Feature_000  3014.4529  3011.4900    73.6218       0.3829  45.2300   
3     4  Feature_044    10.0312    10.0348     0.1750       0.0638   0.1143   
4     5  Feature_051   157.4210   183.3182    60.9251       0.0638  41.7907   
5     6  Feature_174     0.3201     0.3238     0.0712       0.0638   0.0469   

   Flagged_Count  Flagged_Pct  
0             34       2.1795  
1             12       0.9181  
2             12       0.7687  
3              1       0.0639  
4              0       0.0000  
5              0       0.0000  
Selected Features: ['Feature_000', 'Feature_044', 'Feature_051', 'Feature_064', 'Feature_174', 'Feature_556']
Highest Missing %: 16.59 ( Feature_556 )
Highest Flagged % Feature: Feature_064
Total Flagged Observations: 59
V

In [18]:
feature_names = [f"Feature_{i:03d}" for i in range(raw_measurements.shape[1])]

df_all = pd.DataFrame(raw_measurements, columns=feature_names)
df_all.index.name = "Record_ID"
df_all["Result"] = labels
df_all["Outcome"] = np.where(labels == -1, "Pass", "Fail")

pass_count = (labels == -1).sum()
fail_count = (labels == 1).sum()
per_group = min(20 + ID_SUM, pass_count, fail_count)

rng2 = np.random.default_rng(RANDOM_SEED)
pass_ids = rng2.choice(np.flatnonzero(labels == -1), size=per_group, replace=False)
fail_ids = rng2.choice(np.flatnonzero(labels == 1), size=per_group, replace=False)
sample_ids = np.sort(np.concatenate([pass_ids, fail_ids]))

ex2_indices = np.sort(rng2.choice(usable_indices, size=5, replace=False))
ex2_names = [feature_names[i] for i in ex2_indices]

print("Pass Count:", pass_count, "| Fail Count:", fail_count)
print("Records Per Outcome:", per_group)
print("Balanced Sample Size:", len(sample_ids))
print("Selected Features:", ex2_names)

Pass Count: 1463 | Fail Count: 104
Records Per Outcome: 47
Balanced Sample Size: 94
Selected Features: ['Feature_088', 'Feature_132', 'Feature_281', 'Feature_469', 'Feature_489']


In [19]:
full_medians = df_all[ex2_names].median()

balanced = df_all.loc[sample_ids, ex2_names + ["Result", "Outcome"]].copy()
missing_before = balanced[ex2_names].isna().sum().sum()
balanced[ex2_names] = balanced[ex2_names].fillna(full_medians)

balanced_path = OUTPUT_DIR / "balanced_sample.csv"
balanced.reset_index().to_csv(balanced_path, index=False)

print("Missing Before Imputation:", missing_before)
print("Missing After Imputation:", balanced[ex2_names].isna().sum().sum())
print("Outcome Counts:\n", balanced["Outcome"].value_counts())
print("File Created:", balanced_path.name, balanced_path.is_file())

Missing Before Imputation: 0
Missing After Imputation: 0
Outcome Counts:
 Outcome
Pass    47
Fail    47
Name: count, dtype: int64
File Created: balanced_sample.csv True


In [20]:
grouped = balanced.groupby("Outcome")[ex2_names].agg(
    ["count", "mean", "median", "std"]
)

fail_vals = balanced.loc[balanced["Outcome"] == "Fail", ex2_names].to_numpy()
pass_vals = balanced.loc[balanced["Outcome"] == "Pass", ex2_names].to_numpy()

n_f, n_p = len(fail_vals), len(pass_vals)
mean_f, mean_p = fail_vals.mean(axis=0), pass_vals.mean(axis=0)
sd_f, sd_p = fail_vals.std(axis=0, ddof=1), pass_vals.std(axis=0, ddof=1)

mean_diff = mean_f - mean_p
pooled_sd = np.sqrt(((n_f - 1) * sd_f**2 + (n_p - 1) * sd_p**2) / (n_f + n_p - 2))
smd = np.divide(mean_diff, pooled_sd, out=np.zeros_like(mean_diff), where=pooled_sd > 0)

comparison = pd.DataFrame({
    "Feature": ex2_names,
    "Fail_Mean": mean_f,
    "Pass_Mean": mean_p,
    "Mean_Difference": mean_diff,
    "Abs_Mean_Difference": np.abs(mean_diff),
    "Pooled_SD": pooled_sd,
    "Standardized_Mean_Difference": smd,
    "Abs_Standardized_Mean_Difference": np.abs(smd),
}).sort_values("Abs_Standardized_Mean_Difference", ascending=False).reset_index(drop=True)
comparison.insert(0, "Rank", comparison.index + 1)

comparison_path = OUTPUT_DIR / "feature_comparison.csv"
comparison.to_csv(comparison_path, index=False)


pd_mean = balanced.groupby("Outcome")[ex2_names].mean()
pd_median = balanced.groupby("Outcome")[ex2_names].median()
pd_std = balanced.groupby("Outcome")[ex2_names].std()

stats_match = (
    np.allclose(pd_mean.loc["Fail"], mean_f) and np.allclose(pd_mean.loc["Pass"], mean_p)
    and np.allclose(pd_median.loc["Fail"], np.median(fail_vals, axis=0))
    and np.allclose(pd_median.loc["Pass"], np.median(pass_vals, axis=0))
    and np.allclose(pd_std.loc["Fail"], sd_f) and np.allclose(pd_std.loc["Pass"], sd_p)
)

print(grouped.round(4).T)
print(comparison.round(4))
print("Balanced Sample Size:", len(balanced))
print("Selected Features:", ex2_names)
print("Highest-Ranked Feature:", comparison.loc[0, "Feature"])
print("Abs Standardized Mean Difference:",
      f"{comparison.loc[0, 'Abs_Standardized_Mean_Difference']:.4f}")
print("NumPy-Pandas Statistics Match:", stats_match)
print("File Created:", comparison_path.name, comparison_path.is_file())

Outcome                  Fail       Pass
Feature_088 count     47.0000    47.0000
            mean    1817.0501  1814.0028
            median  1823.8228  1819.2784
            std       50.4653    49.3619
Feature_132 count     47.0000    47.0000
            mean       2.3170     2.3230
            median     2.3235     2.3128
            std        0.0600     0.0500
Feature_281 count     47.0000    47.0000
            mean       0.0149     0.0150
            median     0.0136     0.0134
            std        0.0049     0.0051
Feature_469 count     47.0000    47.0000
            mean       5.7527     5.3126
            median     5.6892     5.3391
            std        0.8183     0.9864
Feature_489 count     47.0000    47.0000
            mean     293.7207   264.8104
            median   199.7139   228.2318
            std      235.9944   165.5352
   Rank      Feature  Fail_Mean  Pass_Mean  Mean_Difference  \
0     1  Feature_469     5.7527     5.3126           0.4401   
1     2  Feat

In [21]:
fail_idx = np.flatnonzero(labels == 1)
pass_idx = np.flatnonzero(labels == -1)

def select_ex3_features(seed):
    obs_f = (~np.isnan(raw_measurements[fail_idx])).sum(axis=0)
    obs_p = (~np.isnan(raw_measurements[pass_idx])).sum(axis=0)
    ok = np.zeros(raw_measurements.shape[1], dtype=bool)
    ok[usable_indices] = True
    ok &= (obs_f >= 10) & (obs_p >= 10)
    eligible = np.flatnonzero(ok)
    r = np.random.default_rng(seed)
    chosen = np.sort(r.choice(eligible, size=5, replace=False))
    return eligible, chosen, obs_f, obs_p

eligible, ex3_indices, obs_f, obs_p = select_ex3_features(RANDOM_SEED)
ex3_names = [f"Feature_{i:03d}" for i in ex3_indices]
B = 200 + 10 * SEED_DIGIT

X = raw_measurements[:, ex3_indices]
full_medians = np.nanmedian(X, axis=0)
X_imp = np.where(np.isnan(X), full_medians, X)

Xa_fail, Xa_pass = X[fail_idx], X[pass_idx]
Xi_fail, Xi_pass = X_imp[fail_idx], X_imp[pass_idx]

print("Eligible Features (>=10 observed per group):", len(eligible))
print("Selected Features:", ex3_names)
print("Observed In Fail Group:", obs_f[ex3_indices])
print("Observed In Pass Group:", obs_p[ex3_indices])
print("Bootstrap Repetitions:", B)

Eligible Features (>=10 observed per group): 470
Selected Features: ['Feature_000', 'Feature_051', 'Feature_064', 'Feature_175', 'Feature_558']
Observed In Fail Group: [104 104 104 104 104]
Observed In Pass Group: [1457 1462 1456 1462 1462]
Bootstrap Repetitions: 270


In [22]:
def pooled_sd(Xf, Xp):
    nf = (~np.isnan(Xf)).sum(axis=0)
    npp = (~np.isnan(Xp)).sum(axis=0)
    vf = np.nanvar(Xf, axis=0, ddof=1)
    vp = np.nanvar(Xp, axis=0, ddof=1)
    return np.sqrt(((nf - 1) * vf + (npp - 1) * vp) / (nf + npp - 2))

def safe_ratio(a, b):
    return np.divide(a, b, out=np.zeros_like(a), where=b > 0)

obs_a = np.nanmean(Xa_fail, axis=0) - np.nanmean(Xa_pass, axis=0)
obs_i = Xi_fail.mean(axis=0) - Xi_pass.mean(axis=0)

smd_a = safe_ratio(obs_a, pooled_sd(Xa_fail, Xa_pass))
smd_i = safe_ratio(obs_i, pooled_sd(Xi_fail, Xi_pass))
rank_a = (-np.abs(smd_a)).argsort().argsort() + 1
rank_i = (-np.abs(smd_i)).argsort().argsort() + 1

def run_bootstrap(seed):
    rng = np.random.default_rng(seed)
    n_f, n_p = len(fail_idx), len(pass_idx)
    boot_a = np.empty((B, len(ex3_indices)))
    boot_i = np.empty((B, len(ex3_indices)))
    for b in range(B):
        fr = rng.integers(0, n_f, size=n_f)
        pr = rng.integers(0, n_p, size=n_p)
        boot_a[b] = np.nanmean(Xa_fail[fr], axis=0) - np.nanmean(Xa_pass[pr], axis=0)
        boot_i[b] = Xi_fail[fr].mean(axis=0) - Xi_pass[pr].mean(axis=0)
    return boot_a, boot_i

boot_a, boot_i = run_bootstrap(RANDOM_SEED)

ci_a = np.percentile(boot_a, [2.5, 97.5], axis=0)
ci_i = np.percentile(boot_i, [2.5, 97.5], axis=0)

change = obs_i - obs_a
sign_changed = np.sign(obs_a) != np.sign(obs_i)
rank_changed = rank_a != rank_i

robustness = pd.DataFrame({
    "Feature": ex3_names,
    "Avail_Mean_Diff": obs_a,
    "Imputed_Mean_Diff": obs_i,
    "Change_In_Diff": change,
    "Abs_Change_In_Diff": np.abs(change),
    "Avail_Boot_Mean": boot_a.mean(axis=0),
    "Avail_CI_Lower": ci_a[0],
    "Avail_CI_Upper": ci_a[1],
    "Avail_CI_Excludes_Zero": (ci_a[0] > 0) | (ci_a[1] < 0),
    "Imputed_Boot_Mean": boot_i.mean(axis=0),
    "Imputed_CI_Lower": ci_i[0],
    "Imputed_CI_Upper": ci_i[1],
    "Imputed_CI_Excludes_Zero": (ci_i[0] > 0) | (ci_i[1] < 0),
    "Avail_Rank": rank_a,
    "Imputed_Rank": rank_i,
    "Sign_Changed": sign_changed,
    "Rank_Changed": rank_changed,
})

robustness_path = OUTPUT_DIR / "robustness_comparison.csv"
robustness.to_csv(robustness_path, index=False)

top = change.argmax() if change.max() >= -change.min() else change.argmin()
largest_name = ex3_names[top]
largest_change = change[top]

print(robustness.round(4).T)
print("Selected Features:", ex3_names)
print("Bootstrap Repetitions:", B)
print("Largest Change In Mean Difference:",
      f"{largest_change:.4f}", "(", largest_name, ")")
print("Features With Changed Sign:",
      [n for n, s in zip(ex3_names, sign_changed) if s] or "None")
print("Features With Changed Ranking:",
      [n for n, r in zip(ex3_names, rank_changed) if r] or "None")
print("File Created:", robustness_path.name, robustness_path.is_file())

                                    0            1            2            3  \
Feature                   Feature_000  Feature_051  Feature_064  Feature_175   
Avail_Mean_Diff               -7.4211       1.4688       1.5273         0.02   
Imputed_Mean_Diff             -7.4069        1.451       1.5293         0.02   
Change_In_Diff                 0.0142      -0.0178        0.002          0.0   
Abs_Change_In_Diff             0.0142       0.0178        0.002          0.0   
Avail_Boot_Mean               -7.5471       1.1079       1.4454       0.0199   
Avail_CI_Lower               -23.9904     -12.8723       0.2042      -0.0011   
Avail_CI_Upper                10.5512      14.8924       2.9069       0.0423   
Avail_CI_Excludes_Zero          False        False         True        False   
Imputed_Boot_Mean             -7.5339       1.0897       1.4473       0.0199   
Imputed_CI_Lower             -23.9893     -12.9108       0.2063      -0.0011   
Imputed_CI_Upper                10.56   

In [23]:
eligible2, ex3_indices2, _, _ = select_ex3_features(RANDOM_SEED)
boot_a2, boot_i2 = run_bootstrap(RANDOM_SEED)

reproducible = (
    np.array_equal(ex3_indices, ex3_indices2)
    and np.array_equal(boot_a, boot_a2)
    and np.array_equal(boot_i, boot_i2)
)
assert reproducible, "Results changed with the same seed."

reloaded = pd.read_csv(robustness_path)
pd.testing.assert_frame_equal(
    reloaded, robustness, check_dtype=False,
    check_exact=False, rtol=1e-10, atol=1e-10
)

sign_list = [n for n, s in zip(ex3_names, sign_changed) if s]
rank_list = [n for n, r in zip(ex3_names, rank_changed) if r]

lines = [
    "SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED}",
    f"Selected Features: {', '.join(ex3_names)}",
    f"Bootstrap Repetitions: {B}",
    f"Largest Change In Mean Difference: {largest_change:.4f} ({largest_name})",
    f"Features With Changed Sign: {', '.join(sign_list) if sign_list else 'None'}",
    f"Features With Changed Ranking: {', '.join(rank_list) if rank_list else 'None'}",
    f"Reproducibility Verified: {reproducible}",
    "",
    "Per-Feature Results (Fail minus Pass):",
]
for _, r in robustness.iterrows():
    lines.append(
        f"- {r['Feature']}: available-case {r['Avail_Mean_Diff']:.4f} "
        f"[{r['Avail_CI_Lower']:.4f}, {r['Avail_CI_Upper']:.4f}]; "
        f"imputed {r['Imputed_Mean_Diff']:.4f} "
        f"[{r['Imputed_CI_Lower']:.4f}, {r['Imputed_CI_Upper']:.4f}]; "
        f"rank {r['Avail_Rank']} -> {r['Imputed_Rank']}"
    )
lines += [
    "",
    "Interpretation Notes:",
    "- Intervals that include zero do not show a stable group difference.",
    "- Raw mean differences depend on feature scale; rankings use standardized differences.",
    "- Median imputation can shrink or shift differences compared with available-case results.",
    "- Group differences do not establish failure causes.",
]

report_path3 = OUTPUT_DIR / "sensitivity_report.txt"
report_path3.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Reproducibility Verified:", reproducible)
print("Exported CSV Reloaded And Matches:", True)
for p in [robustness_path, report_path3]:
    print(p.name, "Created:", p.is_file())

Reproducibility Verified: True
Exported CSV Reloaded And Matches: True
robustness_comparison.csv Created: True
sensitivity_report.txt Created: True
